# moe-phone G2 + G3 on Kaggle — OLMoE-1B-7B

## ⚠ Before you press anything

Open the **right-hand panel → Session options** and set:

- **Accelerator: `GPU T4 x2`** — the default is `None`, and with `None` this notebook stops at
  the third cell. Wait for the session to restart after changing it.
- **Internet: `On`** — the model and the corpus are downloaded.

Then **Run All** (or *Save Version → Save & Run All*). The accelerator must be set **before** the
run starts; changing it afterwards does not re-run anything.

This notebook is **generated** by `moe-phone/kaggle/build_notebook.py` from
`moe-phone/gates/traces_sparsity.py` and `moe-phone/gates/cache_sim.py`.
Do not edit the code cells here — edit the repo files and regenerate, or the
Kaggle result stops being a result of the code in the repository.

Every step below runs through `subprocess` and **raises on a non-zero exit**, so
*Run All* stops at the first failure instead of producing an empty archive. In
particular the self-test must print `0 failure(s)`; if it does not, nothing
after it means anything.

What comes back:

| gate | question | artifact |
|---|---|---|
| **G2** | which experts the model's own router picks, per layer and token; whether the next layer's experts can be predicted without training | `traces_OLMoE-1B-7B-0924.npz`, then LRU / Belady / no-locality-floor hit rates |
| **G3** | how much gate-first sparsity the model tolerates, against the Q4_0 quantization floor as the pre-registered Tier-A margin | `g3_OLMoE-1B-7B-0924.json` |

T4s have no bfloat16, so the reference runs in float16 and the artifact records
that. When the last cell finishes, download `/kaggle/working/moe_phone_out.zip`
from the **Output** panel and unpack it into `moe-phone/results/<today>/`.


In [ ]:
%%writefile traces_sparsity.py
"""
G2 + G3 — routing traces, training-free lookahead, and gate-first sparsity
fidelity for STOCK MoE checkpoints (no weights changed, nothing trained).

Runs on a GPU: Kaggle (T4 x2) for models up to ~16B; a rented H100 (80 GB)
for 30B-class 16-bit references. `--selftest` runs on CPU with tiny random
models and must pass before any paid GPU time is spent.

What it measures, per model:
  G2  traces   — the experts the model's OWN router selects, per MoE layer and
                 token (saved to traces_<model>.npz for gates/cache_sim.py).
      lookahead — training-free next-layer expert prediction (Fate, arXiv
                 2502.12224): apply layer l+1's router to layer l's router
                 input; report recall of the true top-k.
  G3  sparsity — gate-first sparsity: for each routed expert, compute the gate
                 activation a = act(gate(x)) exactly, zero neurons with
                 |a| < t_layer, and skip their up/down rows:
                     out = down( (a * [|a| >= t]) * up(x) )
                 This is the read pattern a phone engine can execute without a
                 predictor (read gate rows, then only the needed up/down rows).
                 t_layer is calibrated per layer to a target density on a
                 CALIBRATION corpus (wikitext-2 train) and evaluated on a
                 DISJOINT corpus (wikitext-2 test) — never on a task
                 (CLAUDE.md §6.4).
      fidelity — per token, under teacher forcing, vs the unmodified model:
                 KL(p_ref || p_sparse) mean and p99, top-1 flip rate, and the
                 change in top-1 calibration (ECE, 15 equal-width bins as in
                 Guo et al., ICML 2017) — "does it still know when it is wrong".
      floor   — the Tier-A margin (ESTIMAND.md §3): the KL that Q4_0, the
                 published 4-bit format, adds relative to the 16-bit model.
                 Q4_0 is transcribed from ggml's reference quantizer (see
                 q4_0_fake_quant). Routers, embeddings and lm_head are left
                 unquantized; that choice is recorded in the artifact.

Supported layouts: MoE blocks exposing `.gate` (router) and `.experts` as a
ModuleList of MLPs with (gate_proj, up_proj, down_proj) or (w1, w3, w2) and
`.act_fn` — Qwen2/3-MoE, OLMoE, Mixtral, DeepSeek-V2 (remote code). Fused-
expert implementations are refused, not approximated. Pinned:
transformers==4.56.2 (the version the self-test was run with).

Run:
  python traces_sparsity.py --selftest
  python traces_sparsity.py --model allenai/OLMoE-1B-7B-0924 --windows 64 --seq-len 512
  python traces_sparsity.py --model Qwen/Qwen3-30B-A3B --windows 64 --seq-len 512
"""
import argparse
import datetime
import json
import math
import os
import re
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F

DENSITIES = [1.0, 0.5, 0.3, 0.2, 0.1]
ECE_BINS = 15


# --------------------------------------------------------------------------
# Q4_0 — transcribed from ggml/src/ggml-quants.c, quantize_row_q4_0_ref:
#   for each block of QK4_0 = 32 consecutive weights:
#     amax = max_j |x_j|, max = the x_j attaining it (signed)
#     d  = max / -8          (stored as fp16)
#     id = d ? 1/d : 0
#     q_j = MIN(15, (int8_t)(x_j * id + 8.5f))       (truncation toward zero)
#   dequantized value = (q_j - 8) * d
# Symbol map: x -> w (blocks along the input dimension), d -> scale, q -> q.
# Verify against the ggml commit you compare with before publication.
# --------------------------------------------------------------------------
def q4_0_fake_quant(w):
    shape = w.shape
    x = w.detach().float().reshape(-1, 32)
    idx = x.abs().argmax(dim=1, keepdim=True)
    mx = torch.gather(x, 1, idx)
    d = (mx / -8.0).half().float()
    inv = torch.where(d != 0, 1.0 / d, torch.zeros_like(d))
    q = torch.clamp(torch.trunc(x * inv + 8.5), max=15)
    return ((q - 8.0) * d).reshape(shape).to(w.dtype)


def q4_0_scalar_reference(block):
    """Direct scalar transcription of the ggml loop, for the self-test."""
    amax, mx = 0.0, 0.0
    for v in block:
        if amax < abs(v):
            amax, mx = abs(v), v
    d = float(np.float16(mx / -8.0))
    inv = 1.0 / d if d else 0.0
    out = []
    for v in block:
        q = min(15, int(np.float32(v) * np.float32(inv) + np.float32(8.5)))
        out.append((q - 8) * d)
    return out


# --------------------------------------------------------------------------
# Model structure
# --------------------------------------------------------------------------
def moe_blocks(model):
    blocks = []
    for name, mod in model.named_modules():
        experts, gate = getattr(mod, "experts", None), getattr(mod, "gate", None)
        if isinstance(experts, torch.nn.ModuleList) and gate is not None:
            m = re.search(r"layers\.(\d+)\.", name + ".")
            if m:
                blocks.append((int(m.group(1)), name, mod))
    if not blocks:
        raise SystemExit("No MoE block with `.gate` and a ModuleList `.experts` found. This is "
                         "probably a fused-expert implementation; pin transformers==4.56.2.")
    return sorted(blocks, key=lambda b: b[0])


def mlp_parts(mlp):
    for g, u, d in (("gate_proj", "up_proj", "down_proj"), ("w1", "w3", "w2")):
        if all(hasattr(mlp, x) for x in (g, u, d)) and hasattr(mlp, "act_fn"):
            return getattr(mlp, g), getattr(mlp, u), getattr(mlp, d), mlp.act_fn
    raise SystemExit(f"Unsupported expert MLP layout: {type(mlp).__name__}")


def top_k_of(model):
    c = model.config
    for k in ("num_experts_per_tok", "moe_k", "experts_per_token"):
        if getattr(c, k, None):
            return int(getattr(c, k))
    raise SystemExit("config has no top-k field")


def num_experts_of(block):
    return len(block.experts)


# --------------------------------------------------------------------------
# Instrumentation
# --------------------------------------------------------------------------
class State:
    def __init__(self, layers):
        self.mode = "off"          # off | calib | apply
        self.trace = False
        self.thr = {}              # layer -> threshold (current density)
        self.samples = {l: [] for l in layers}
        self.kept = {l: 0 for l in layers}
        self.total = {l: 0 for l in layers}
        self.cur = {}              # layer -> selected expert ids [T, k] (cpu int16)
        self.inp = {}              # layer -> router input [T, H] (device)
        self.per_call = 4096


def instrument(model, S, top_k):
    blocks = moe_blocks(model)
    for layer, _, block in blocks:
        for mlp in block.experts:
            G, U, D, act = mlp_parts(mlp)
            orig = mlp.forward

            def fwd(x, *args, _orig=orig, _G=G, _U=U, _D=D, _act=act, _l=layer, **kw):
                if S.mode == "off":
                    return _orig(x, *args, **kw)
                a = _act(_G(x))
                if S.mode == "calib":
                    flat = a.detach().abs().float().flatten()
                    if flat.numel() > S.per_call:
                        flat = flat[torch.randint(0, flat.numel(), (S.per_call,), device=flat.device)]
                    S.samples[_l].append(flat.cpu())
                    return _D(a * _U(x))
                m = a.abs() >= S.thr[_l]
                S.kept[_l] += int(m.sum())
                S.total[_l] += m.numel()
                return _D((a * m) * _U(x))

            mlp.forward = fwd

        def pre(mod, args, _l=layer):
            if S.trace:
                x = args[0]
                S.inp[_l] = x.detach().reshape(-1, x.shape[-1])

        def post(mod, args, out, _l=layer):
            if S.trace:
                ids = out[0] if isinstance(out, tuple) else torch.topk(out.float(), top_k, dim=-1).indices
                S.cur[_l] = ids.detach().reshape(-1, ids.shape[-1]).to(torch.int16).cpu()

        block.gate.register_forward_pre_hook(pre)
        block.gate.register_forward_hook(post)
    return blocks


def router_logits(gate, x):
    """Apply a router's weight matrix to an arbitrary activation.

    `x` is moved to the weight's DEVICE as well as its dtype. The Fate-style
    lookahead (arXiv 2502.12224) deliberately applies layer l+1's gate to layer
    l's router input, and under `device_map="auto"` those two layers can sit on
    different GPUs. At the shard boundary this raised, on Kaggle's 2x T4
    (2026-09-14):

        RuntimeError: Expected all tensors to be on the same device, but got
        mat2 is on cuda:1, different from other tensors on cuda:0

    The CPU self-test cannot reach this path, because it has one device. That
    is a gap in the self-test, not a reason the check is unnecessary: see
    selftest()'s cross-device case, which runs only when a second device exists.
    """
    W = gate.weight
    return F.linear(x.to(device=W.device, dtype=W.dtype), W).float()


def lookahead_supported(model):
    m = getattr(model.config, "topk_method", "greedy")
    return m in (None, "greedy")


# --------------------------------------------------------------------------
# Metrics
# --------------------------------------------------------------------------
def token_kl(logp_ref, logp):
    return (logp_ref.exp() * (logp_ref - logp)).sum(-1)


def nll_sum(logp, labels):
    """Summed teacher-forced negative log-likelihood of the true next token.

    Perplexity = exp(sum(nll) / n_tokens). Reported because the activation-
    sparsity literature justifies its sparsity levels with PERPLEXITY curves
    (TEAL, arXiv 2408.14690, reports perplexity vs sparsity; it never reports
    KL divergence or top-1 flip rate). ESTIMAND.md §3 sets our Tier-A margin in
    KL and flip rate instead, which is a far stricter bar: a model can change
    its argmax on a quarter of tokens while perplexity moves very little,
    because perplexity only scores the probability mass on the TRUE token.
    Without perplexity in our own artifact we cannot tell whether we DISAGREE
    with the published results or merely MEASURE something else — so both are
    reported, and any comparison to published sparsity levels must use this one.
    """
    return float(-logp[:-1].gather(-1, labels.unsqueeze(-1)).sum()), labels.numel()


class Calib:
    def __init__(self):
        self.conf, self.correct = [], []

    def add(self, logp, labels):
        c, pred = logp[:-1].exp().max(-1)
        self.conf.append(c.float().cpu())
        self.correct.append((pred == labels).float().cpu())

    def ece(self):
        c, k = torch.cat(self.conf), torch.cat(self.correct)
        e = torch.zeros(())
        for b in range(ECE_BINS):
            lo, hi = b / ECE_BINS, (b + 1) / ECE_BINS
            m = (c > lo) & (c <= hi)
            if m.any():
                e += m.float().mean() * (c[m].mean() - k[m].mean()).abs()
        return float(e)


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
def windows_from_text(tok, split, n, seq_len):
    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
    ids = tok("\n\n".join(ds["text"]), return_tensors="pt").input_ids[0]
    n = min(n, ids.numel() // seq_len)
    return [ids[i * seq_len:(i + 1) * seq_len] for i in range(n)]


# --------------------------------------------------------------------------
# Core run
# --------------------------------------------------------------------------
@torch.no_grad()
def run(model, calib_windows, eval_windows, densities, floor_tokens, device, log=print):
    top_k = top_k_of(model)
    blocks = moe_blocks(model)
    layers = [b[0] for b in blocks]
    S = State(layers)
    instrument(model, S, top_k)
    E = num_experts_of(blocks[0][2])

    def fwd(ids):
        out = model(ids.unsqueeze(0).to(device)).logits[0].float()
        # float16 on GPUs without bfloat16 (Kaggle T4) can overflow; a KL computed
        # from inf/nan logits would be silently meaningless, so stop instead.
        if not torch.isfinite(out).all():
            raise FloatingPointError(f"non-finite logits (mode={S.mode}); rerun with --dtype bfloat16 "
                                     f"on a GPU that supports it, or on an H100")
        return out

    # 1. Calibrate thresholds (calibration corpus only).
    S.mode = "calib"
    for w in calib_windows:
        fwd(w)
    thresholds = {}
    for d in densities:
        thresholds[d] = {}
        for l in layers:
            v = torch.cat(S.samples[l]).numpy()
            thresholds[d][l] = 0.0 if d >= 1.0 else float(np.quantile(v, 1.0 - d))
    S.samples = {l: [] for l in layers}

    traces = {l: [] for l in layers}
    look_recall, look_n = {}, {}
    acc = {d: {"kl": [], "flip": 0, "n": 0, "calib": Calib(), "nll": 0.0, "nll_n": 0}
           for d in densities}
    ref_nll, ref_nll_n = 0.0, 0
    ref_calib = Calib()
    floor_store = []
    stored = 0
    for wi, w in enumerate(eval_windows):
        # 2. Reference pass with tracing.
        S.mode, S.trace = "off", True
        ref = fwd(w)
        S.trace = False
        logp_ref = F.log_softmax(ref, -1)
        labels = w[1:].to(device)
        ref_calib.add(logp_ref, labels)
        _s, _n = nll_sum(logp_ref, labels)
        ref_nll += _s; ref_nll_n += _n
        for l in layers:
            traces[l].append(S.cur[l].numpy())
        if lookahead_supported(model):
            for (l, _, b), (l2, _, b2) in zip(blocks[:-1], blocks[1:]):
                pred = torch.topk(router_logits(b2.gate, S.inp[l]), top_k, -1).indices.cpu()
                act = S.cur[l2].long()
                hit = (pred.unsqueeze(-1) == act.unsqueeze(-2)).any(-1).float().mean(-1)
                look_recall[l2] = look_recall.get(l2, 0.0) + float(hit.sum())
                look_n[l2] = look_n.get(l2, 0) + hit.numel()
        if stored < floor_tokens:
            floor_store.append((w, logp_ref.half().cpu()))
            stored += w.numel()
        # 3. Sparse passes.
        for d in densities:
            S.mode, S.thr = "apply", thresholds[d]
            lp = F.log_softmax(fwd(w), -1)
            kl = token_kl(logp_ref, lp)
            a = acc[d]
            a["kl"].append(kl.cpu())
            a["flip"] += int((lp.argmax(-1) != logp_ref.argmax(-1)).sum())
            a["n"] += kl.numel()
            a["calib"].add(lp, labels)
            _s, _n = nll_sum(lp, labels)
            a["nll"] += _s; a["nll_n"] += _n
        S.mode = "off"
        log(f"  window {wi + 1}/{len(eval_windows)}")

    realized = {}
    # kept/total were accumulated across all densities; recompute per density cleanly.
    results = {"top_k": top_k, "num_experts": E, "moe_layers": layers,
               "ref_ece": ref_calib.ece(),
               "ref_perplexity": math.exp(ref_nll / ref_nll_n), "densities": {}}
    for d in densities:
        a = acc[d]
        kl = torch.cat(a["kl"])
        results["densities"][str(d)] = {
            "kl_mean": float(kl.mean()), "kl_p99": float(torch.quantile(kl, 0.99)),
            "flip_rate": a["flip"] / a["n"], "ece": a["calib"].ece(), "tokens": a["n"],
            "perplexity": math.exp(a["nll"] / a["nll_n"]),
            "perplexity_ratio_vs_dense": math.exp(a["nll"] / a["nll_n"]) / math.exp(ref_nll / ref_nll_n),
            "thresholds": {str(l): thresholds[d][l] for l in layers}}
    # Realized density on eval data, one clean pass per density on the first window.
    for d in densities:
        S.kept = {l: 0 for l in layers}
        S.total = {l: 0 for l in layers}
        S.mode, S.thr = "apply", thresholds[d]
        fwd(eval_windows[0])
        realized[str(d)] = {str(l): S.kept[l] / max(1, S.total[l]) for l in layers}
    S.mode = "off"
    results["realized_density_first_window"] = realized
    results["lookahead_recall"] = ({str(l): look_recall[l] / look_n[l] for l in look_recall}
                                   if look_recall else "unsupported for this router")
    return results, {l: np.concatenate(traces[l]) for l in layers}, floor_store, S


@torch.no_grad()
def format_floor(model, floor_store, device):
    """Quantize linear weights in place to Q4_0 (irreversible — run last)."""
    skipped = []
    for name, mod in model.named_modules():
        if isinstance(mod, torch.nn.Linear):
            if name.endswith("lm_head") or name.endswith(".gate") or mod.in_features % 32:
                skipped.append(name)
                continue
            mod.weight.data = q4_0_fake_quant(mod.weight.data)
    kls, flips, n = [], 0, 0
    for w, lr in floor_store:
        lp = F.log_softmax(model(w.unsqueeze(0).to(device)).logits[0].float(), -1)
        lr = lr.to(device).float()
        kl = token_kl(lr, lp)
        kls.append(kl.cpu())
        flips += int((lp.argmax(-1) != lr.argmax(-1)).sum())
        n += kl.numel()
    kl = torch.cat(kls)
    return {"format": "Q4_0", "kl_mean": float(kl.mean()), "kl_p99": float(torch.quantile(kl, 0.99)),
            "flip_rate": flips / n, "tokens": n,
            "left_unquantized": "lm_head, routers (*.gate), layers with in_features % 32 != 0",
            "n_linear_skipped": len(skipped)}


# --------------------------------------------------------------------------
# Self-test — every assertion is fixed by definition, not by a result
# --------------------------------------------------------------------------
def selftest():
    import transformers
    from transformers import (MixtralConfig, MixtralForCausalLM, OlmoeConfig, OlmoeForCausalLM,
                              Qwen3MoeConfig, Qwen3MoeForCausalLM)
    print("transformers", transformers.__version__, "torch", torch.__version__)
    failures = []

    def check(cond, msg):
        print(("PASS " if cond else "FAIL ") + msg)
        if not cond:
            failures.append(msg)

    rng = np.random.default_rng(0)
    for _ in range(50):
        blk = (rng.standard_normal(32) * rng.uniform(0.01, 3)).astype(np.float32)
        ref = np.array(q4_0_scalar_reference(blk.tolist()), dtype=np.float32)
        got = q4_0_fake_quant(torch.from_numpy(blk)).numpy()
        if not np.allclose(ref, got, atol=0, rtol=0):
            check(False, "Q4_0 vectorized == scalar ggml transcription")
            break
    else:
        check(True, "Q4_0 vectorized == scalar ggml transcription (50 random blocks)")
    blk = np.zeros(32, dtype=np.float32)
    blk[0], blk[1], blk[2] = -4.0, 1.0, 0.3
    got = q4_0_fake_quant(torch.from_numpy(blk)).numpy()
    check(got[0] == -4.0 and got[1] == 1.0 and got[2] == 0.5,
          "Q4_0 hand block: d=0.5 -> [-4, 1, 0.3] dequantize to [-4, 1, 0.5]")

    base = dict(vocab_size=96, hidden_size=64, num_hidden_layers=3, num_attention_heads=4,
                max_position_embeddings=128)
    models = [
        ("qwen3_moe", Qwen3MoeForCausalLM, Qwen3MoeConfig(**base, intermediate_size=128, moe_intermediate_size=32,
                                                         num_key_value_heads=2, head_dim=16, num_experts=8,
                                                         num_experts_per_tok=2)),
        ("olmoe", OlmoeForCausalLM, OlmoeConfig(**base, intermediate_size=32, num_key_value_heads=4,
                                                num_experts=8, num_experts_per_tok=2)),
        ("mixtral", MixtralForCausalLM, MixtralConfig(**base, intermediate_size=32, num_key_value_heads=2,
                                                      num_local_experts=4, num_experts_per_tok=2)),
    ]
    for tag, cls, cfg in models:
        torch.manual_seed(0)
        model = cls(cfg).eval()
        g = torch.Generator().manual_seed(1)
        wins = [torch.randint(0, cfg.vocab_size, (32,), generator=g) for _ in range(4)]
        with torch.no_grad():
            plain = model(wins[0].unsqueeze(0)).logits[0].float()
            out = model(wins[0].unsqueeze(0), output_router_logits=True)
            router = [r.float() for r in out.router_logits]
        res, traces, floor_store, S = run(model, wins[:2], wins[2:], DENSITIES, 64, "cpu", log=lambda *_: None)
        with torch.no_grad():
            S.mode, S.thr = "apply", {l: 0.0 for l in S.thr}
            patched = model(wins[0].unsqueeze(0)).logits[0].float()
            S.mode = "off"
        check(torch.equal(plain, patched), f"[{tag}] density 1.0 (threshold 0) is bit-identical to the unmodified model")
        check(res["densities"]["1.0"]["kl_mean"] == 0.0 and res["densities"]["1.0"]["flip_rate"] == 0.0,
              f"[{tag}] KL and flip rate are exactly 0 at density 1.0")
        k, E = res["top_k"], res["num_experts"]
        ok = all(t.shape[1] == k and t.min() >= 0 and t.max() < E for t in traces.values())
        check(ok, f"[{tag}] traces have shape [T,{k}] and ids in [0,{E})")
        # Traced ids must equal the model's own router decisions (external reference).
        S.trace = True
        with torch.no_grad():
            model(wins[0].unsqueeze(0))
        S.trace = False
        same = all(torch.equal(torch.sort(S.cur[l].long(), -1).values,
                               torch.sort(torch.topk(router[i], k, -1).indices, -1).values)
                   for i, l in enumerate(sorted(S.cur)))
        check(same, f"[{tag}] traced experts == top-k of the model's own router_logits")
        # Lookahead machinery: predicting layer l with ITS OWN router must give recall 1.
        blocks = moe_blocks(model)
        l0, _, b0 = blocks[0]
        own = torch.topk(router_logits(b0.gate, S.inp[l0]), k, -1).indices
        check(torch.equal(torch.sort(own, -1).values, torch.sort(S.cur[l0].long(), -1).values),
              f"[{tag}] router applied to its own captured input reproduces its selection")
        lr = res["lookahead_recall"]
        check(isinstance(lr, dict) and all(0.0 <= v <= 1.0 for v in lr.values()),
              f"[{tag}] lookahead recall is a proportion")
        check(res["densities"]["0.1"]["kl_mean"] > 0.0, f"[{tag}] density 0.1 changes the output (KL > 0)")
        fl = format_floor(model, floor_store, "cpu")
        check(fl["kl_mean"] > 0.0 and fl["tokens"] > 0, f"[{tag}] Q4_0 floor is measurable (KL > 0)")

    # Cross-device lookahead. The Fate lookahead applies layer l+1's gate to
    # layer l's input, and under device_map="auto" those layers can be on
    # different GPUs; that combination raised a RuntimeError on Kaggle's 2x T4
    # on 2026-09-14 while this self-test passed, because one device cannot
    # exercise it. Run it whenever a second device exists, and SAY SO when it
    # cannot be run rather than reporting a pass that did not happen.
    n_dev = torch.cuda.device_count()
    if n_dev >= 2:
        gate = torch.nn.Linear(8, 4, bias=False).to("cuda:1")
        x = torch.randn(3, 8, device="cuda:0")
        try:
            out = router_logits(gate, x)
            check(out.shape == (3, 4), "cross-device lookahead: gate on cuda:1, input on cuda:0")
        except RuntimeError as e:
            check(False, f"cross-device lookahead raised {type(e).__name__}: {e}")
    else:
        print(f"SKIP  cross-device lookahead: needs 2 CUDA devices, found {n_dev}. "
              f"This is the path that failed on Kaggle 2x T4 and it is NOT covered here.")

    print(f"\n{len(failures)} failure(s)")
    return 1 if failures else 0


# --------------------------------------------------------------------------
def out_dir(arg):
    if arg:
        os.makedirs(arg, exist_ok=True)
        return arg
    try:
        sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
        from _paths import write_path
        return os.path.dirname(write_path("_dir_probe"))
    except ImportError:
        d = os.path.join("moe_phone_out", datetime.date.today().isoformat())
        print(f"_paths not importable (running outside the repo); writing to {d}")
        os.makedirs(d, exist_ok=True)
        return d


def main():
    p = argparse.ArgumentParser(description="G2+G3 on a stock MoE checkpoint")
    p.add_argument("--selftest", action="store_true")
    p.add_argument("--model")
    p.add_argument("--windows", type=int, default=64, help="evaluation windows (wikitext-2 test)")
    p.add_argument("--calib-windows", type=int, default=16, help="calibration windows (wikitext-2 train)")
    p.add_argument("--seq-len", type=int, default=512)
    p.add_argument("--floor-tokens", type=int, default=8192)
    p.add_argument("--densities", default=",".join(str(d) for d in DENSITIES))
    p.add_argument("--dtype", default="auto", choices=["auto", "bfloat16", "float16"])
    p.add_argument("--trust-remote-code", action="store_true")
    p.add_argument("--load-in-4bit", action="store_true",
                   help="load the checkpoint with bitsandbytes NF4. Needed to reach a "
                        "30B-class MoE on 2x16GB, which is what closing S9 requires: the "
                        "transfer assumption is about EXPERT COUNT, and every model with a "
                        "materially different E is too large for these GPUs at 16-bit. "
                        "NOTE this quantises the ROUTER too, so routing may differ slightly "
                        "from the 16-bit model -- which is a confound for a fidelity number "
                        "and NOT one for a routing trace, since the engine itself runs "
                        "quantised. The artifact records the flag either way.")
    p.add_argument("--no-floor", action="store_true")
    p.add_argument("--out-dir", default=None)
    a = p.parse_args()
    if a.selftest:
        sys.exit(selftest())
    if not a.model:
        p.error("--model is required unless --selftest")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    cuda = torch.cuda.is_available()
    # torch.cuda.is_bf16_supported() returns True on a Tesla T4, because PyTorch
    # EMULATES bfloat16 there by upcasting to fp32. That is numerically fine but
    # several times slower, and it silently contradicts protocols/RUNBOOK.md,
    # which states "T4s have no bfloat16, so the reference runs in float16" — a
    # documented behaviour that the code did not assert. Compute capability >= 8.0
    # (Ampere) is the test for NATIVE bfloat16. Taken as the minimum over all
    # visible devices, since the model is sharded across them.
    native_bf16 = cuda and min(torch.cuda.get_device_capability(i)[0]
                               for i in range(torch.cuda.device_count())) >= 8
    if a.dtype == "auto":
        dtype = torch.bfloat16 if native_bf16 else torch.float16
    else:
        dtype = getattr(torch, a.dtype)
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(a.model, trust_remote_code=a.trust_remote_code)
    load_kw = {}
    if a.load_in_4bit:
        if not cuda:
            raise SystemExit("--load-in-4bit needs a CUDA device")
        try:
            from transformers import BitsAndBytesConfig
        except ImportError as exc:
            raise SystemExit(f"--load-in-4bit needs bitsandbytes: {exc}")
        load_kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype)
    model = AutoModelForCausalLM.from_pretrained(
        a.model, torch_dtype=dtype, device_map="auto" if cuda else None,
        trust_remote_code=a.trust_remote_code, **load_kw).eval()
    device = next(model.parameters()).device
    calib = windows_from_text(tok, "train", a.calib_windows, a.seq_len)
    evalw = windows_from_text(tok, "test", a.windows, a.seq_len)
    dens = [float(x) for x in a.densities.split(",")]
    if 1.0 not in dens:
        dens = [1.0] + dens
    print(f"{a.model}: {len(calib)} calibration + {len(evalw)} evaluation windows of {a.seq_len}, dtype {dtype}")
    res, traces, floor_store, _ = run(model, calib, evalw, dens, a.floor_tokens, device)
    res["floor"] = None if a.no_floor else format_floor(model, floor_store, device)
    res.update({"model": a.model, "dtype": str(dtype), "bf16_native": native_bf16,
                "load_in_4bit": bool(a.load_in_4bit),
                "n_cuda_devices": torch.cuda.device_count() if cuda else 0,
                "seq_len": a.seq_len,
                "calib_windows": len(calib), "eval_windows": len(evalw),
                "corpus": "wikitext-2-raw-v1 (calibration: train, evaluation: test)",
                "transformers": __import__("transformers").__version__, "torch": torch.__version__,
                "runtime_s": time.time() - t0})
    od = out_dir(a.out_dir)
    tag = a.model.split("/")[-1]
    with open(os.path.join(od, f"g3_{tag}.json"), "w", encoding="utf-8") as f:
        json.dump(res, f, indent=1)
    np.savez_compressed(os.path.join(od, f"traces_{tag}.npz"),
                        num_experts=res["num_experts"], **{f"L{l}": t for l, t in traces.items()})
    print("\n density   KL mean    KL p99   flip    ECE")
    for d, r in res["densities"].items():
        print(f"  {d:>5s}  {r['kl_mean']:.5f}  {r['kl_p99']:.5f}  {r['flip_rate']:.4f}  {r['ece']:.4f}")
    if res["floor"]:
        f = res["floor"]
        print(f"  Q4_0 floor  {f['kl_mean']:.5f}  {f['kl_p99']:.5f}  {f['flip_rate']:.4f}   (Tier-A margin)")
    print(f"reference ECE {res['ref_ece']:.4f}; wrote {od}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cache_sim.py
"""
G2 (offline half) — expert-cache hit rates from routing traces.

Input: the traces_<model>.npz written by traces_sparsity.py — one int16 array
per MoE layer, shape [tokens, top_k], of the experts the model's own router
selected.

TWO AXES, both of which changed a reported number (see the retraction in
POSITION.md F4). Neither is a tuning knob; each names a different machine.

  scope     "global"     one cache shared by all layers.
            "per_layer"  each layer gets its own quota out of the same TOTAL
                         slot budget (split_capacity below). This is what every
                         published expert-streaming engine does, because layer
                         l's experts are never candidates for layer l+1.
            The first release of this file simulated ONLY "global" while
            gates/expert_policy.py's pinning policy was per-layer, so the
            LRU-vs-pinning comparison it printed was between two different
            cache architectures. That comparison is retracted.

  replay    "atomic"     the engine fetches a layer's top-k misses as ONE
                         event, so residency is decided for all k before any
                         eviction. This is what a real engine does.
            "sequential" evict after every single expert access. This lets a
                         miss early in an event evict an expert the SAME event
                         still needs, which is then counted as a miss. It is a
                         simulator artifact, and it inflates LRU's miss count.

Reports, per cache size:

  - LRU hit rate            (what a simple engine gets)
  - Belady-optimal hit rate (the offline optimum: an upper bound for ANY
                             eviction policy, used as the oracle, CLAUDE.md §4.3)
  - no-locality floor       (= cache fraction; what uniform independent routing
                             gives any policy — the value G-ROOF assumed)
  - misses per token        (x expert bytes = flash bytes per token, feeding G-ROOF)

The geometry that decides whether any of this can work is

      rho = (cache slots per layer) / top_k

with a critical threshold at rho = 1, i.e. cache_fraction = k/E: below it a
per-layer cache cannot hold even one token's working set, so LRU degenerates to
the cyclic-scan worst case regardless of how much locality the trace has. Pass
--fractions-around-crit to sweep across it. OLMoE (k/E = 0.125) and
Qwen3-Next-80B-A3B (10/512 = 0.0195) sit on opposite sides at any realistic
cache size, so an OLMoE trace is NOT a conservative proxy for the target model.

Run:
  python moe-phone/gates/cache_sim.py traces_OLMoE-1B-7B-0924.npz --both-scopes
"""
import argparse
import heapq
import json
import os
import sys
from collections import OrderedDict

import numpy as np


def request_stream(traces, num_experts, max_tokens=None):
    """traces: dict layer_index -> int array [T, k]. Returns int64 keys in
    token-major order: key = layer_rank * num_experts + expert."""
    layers = sorted(traces)
    T = min(traces[l].shape[0] for l in layers)
    if max_tokens:
        T = min(T, max_tokens)
    k = traces[layers[0]].shape[1]
    stacked = np.stack([traces[l][:T].astype(np.int64) + r * num_experts
                        for r, l in enumerate(layers)], axis=1)  # [T, L, k]
    return stacked.reshape(-1), T, len(layers), k


def lru_hits(seq, cap):
    cache = OrderedDict()
    hits = 0
    for key in seq.tolist():
        if key in cache:
            hits += 1
            cache.move_to_end(key)
        else:
            if len(cache) >= cap:
                cache.popitem(last=False)
            cache[key] = None
    return hits


def belady_hits(seq, cap):
    """Belady's MIN: on a miss with a full cache, evict the key whose next use
    is farthest in the future. Optimal for any sequence (Belady 1966)."""
    keys = seq.tolist()
    n = len(keys)
    nxt = [0] * n
    last = {}
    for i in range(n - 1, -1, -1):
        nxt[i] = last.get(keys[i], n + i)  # never used again: distinct far-future sentinel
        last[keys[i]] = i
    cache, cur, heap, hits = set(), {}, [], 0
    for i, key in enumerate(keys):
        if key in cache:
            hits += 1
        else:
            if len(cache) >= cap:
                while True:
                    neg, victim = heapq.heappop(heap)
                    if victim in cache and cur[victim] == -neg:
                        cache.remove(victim)
                        break
            cache.add(key)
        cur[key] = nxt[i]
        heapq.heappush(heap, (-nxt[i], key))
    return hits


def event_stream(traces, num_experts, max_tokens=None):
    """Same keys as request_stream, but shaped [T, L, k] so the (token, layer)
    EVENT survives as a unit. One event is one layer's top-k fetch."""
    layers = sorted(traces)
    T = min(traces[l].shape[0] for l in layers)
    if max_tokens:
        T = min(T, max_tokens)
    ev = np.stack([traces[l][:T].astype(np.int64) + r * num_experts
                   for r, l in enumerate(layers)], axis=1)  # [T, L, k]
    return ev, T, len(layers), ev.shape[2]


def split_capacity(cap_total, n_layers):
    """Split a TOTAL slot budget across layers, preserving the total exactly.

    base = cap//L each; the first cap%L layers get one more. Even splitting,
    not proportional-to-traffic: every layer runs on every token, so a layer
    starved of cache stalls the whole token. A proportional split is a tuning
    knob and would need its own justification (CLAUDE.md §6.4).

    The total is preserved so that "10% cache" means the same BYTES under both
    scopes and the two are comparable.
    """
    base, rem = divmod(cap_total, n_layers)
    return [base + (1 if i < rem else 0) for i in range(n_layers)]


def lru_hits_events(ev, cap_total, scope="per_layer", replay="atomic"):
    """LRU over the [T, L, k] event stream. See the module docstring for scope
    and replay; both change the answer and neither has a default that is right
    for every question."""
    T, L, k = ev.shape
    if scope == "per_layer":
        caps = split_capacity(cap_total, L)
        caches = [OrderedDict() for _ in range(L)]
    elif scope == "global":
        caps = [cap_total] * L
        shared = OrderedDict()
        caches = [shared] * L
    else:
        raise ValueError(f"scope must be 'global' or 'per_layer', got {scope!r}")
    if replay not in ("atomic", "sequential"):
        raise ValueError(f"replay must be 'atomic' or 'sequential', got {replay!r}")

    hits = 0
    rows = ev.reshape(T * L, k).tolist()
    layer_of = list(range(L)) * T
    for keys, l in zip(rows, layer_of):
        c, cp = caches[l], caps[l]
        if cp <= 0:
            continue  # a layer with no slots misses everything
        if replay == "atomic":
            # Residency is decided against the cache state at EVENT START, and
            # every hit is promoted before any miss is inserted -- so a miss can
            # never evict an expert this same event still needs.
            miss = []
            for key in keys:
                if key in c:
                    hits += 1
                    c.move_to_end(key)
                else:
                    miss.append(key)
            for key in miss:
                while len(c) >= cp:
                    c.popitem(last=False)
                c[key] = None
        else:
            for key in keys:
                if key in c:
                    hits += 1
                    c.move_to_end(key)
                else:
                    while len(c) >= cp:
                        c.popitem(last=False)
                    c[key] = None
    return hits


def belady_hits_scoped(ev, cap_total, scope="per_layer"):
    """Belady's MIN under the same scope. Per-layer, each layer's subsequence is
    solved independently, which is exactly right: with private quotas the layers
    are independent caching problems.

    Belady is run sequentially in both cases. Unlike LRU it is insensitive to
    the atomic/sequential distinction to first order, because the victim it
    picks is the farthest-next-use and an expert needed later in the SAME event
    has a next use only a few accesses away, so it is never the victim while any
    alternative exists.
    """
    T, L, k = ev.shape
    if scope == "global":
        return belady_hits(ev.reshape(-1), cap_total)
    caps = split_capacity(cap_total, L)
    return sum(belady_hits(ev[:, l, :].reshape(-1), caps[l])
               for l in range(L) if caps[l] > 0)


def shuffle_control(traces, seed=0):
    """Permute the TOKEN order within each layer, independently.

    Preserves each layer's expert-popularity distribution exactly and destroys
    temporal order. The gap between the real trace and this control is the part
    of the hit rate attributable to RECENCY; the gap between this control and
    the uniform control is the part attributable to POPULARITY SKEW.
    """
    rng = np.random.default_rng(seed)
    return {l: t[rng.permutation(t.shape[0])] for l, t in traces.items()}


def uniform_control(traces, num_experts, seed=0):
    """iid uniform top-k per token: no skew and no order. The analytic null --
    LRU on this must equal the cache fraction, which is the recovery gate in
    tests/test_gates.py::test_lru_on_locality_free_routing_equals_the_floor."""
    rng = np.random.default_rng(seed)
    out = {}
    for l, t in traces.items():
        T, k = t.shape
        out[l] = np.stack([rng.choice(num_experts, size=k, replace=False)
                           for _ in range(T)]).astype(t.dtype)
    return out


def window_fetches(ev, cap_total, window, scope="per_layer"):
    """Multi-token verification over the SAME LRU cache.

    A speculative engine runs `window` tokens through one forward pass, so for
    each layer it needs the UNION of those tokens' top-k sets and fetches each
    distinct expert once. This is exactly llama.cpp PR #25294's wave-partitional
    prefill, pointed at decode.

    Returns (fetches, windows): `fetches` counts expert LOADS from flash, which
    is the quantity that costs time. Bytes/token = fetches * expert_bytes /
    tokens_emitted, and tokens_emitted depends on the acceptance rate, so it is
    applied by the caller.

    This composes with the cache rather than multiplying against it: the union
    and the cache exploit the SAME expert reuse, so the saving from widening the
    window is measured here on top of whatever the cache already supplies, not
    assumed to be independent of it.
    """
    T, L, k = ev.shape
    if scope == "per_layer":
        caps = split_capacity(cap_total, L)
        caches = [OrderedDict() for _ in range(L)]
    else:
        caps = [cap_total] * L
        shared = OrderedDict()
        caches = [shared] * L
    fetches = 0
    n_win = 0
    for t0 in range(0, T - window + 1, window):
        n_win += 1
        for l in range(L):
            c, cp = caches[l], caps[l]
            if cp <= 0:
                continue
            need = set(ev[t0:t0 + window, l, :].reshape(-1).tolist())
            miss = [key for key in need if key not in c]
            for key in need:
                if key in c:
                    c.move_to_end(key)
            for key in miss:
                while len(c) >= cp:
                    c.popitem(last=False)
                c[key] = None
            fetches += len(miss)
    return fetches, n_win


def policy_hits(ev, cap_total, policy="lru", scope="per_layer",
                warm=None, pin_frac=0.0, eval_from=0):
    """Deployable online policies, all event-atomic and per-layer by default.

      lru   recency. What a naive engine does.
      lfu   frequency, counted online from the replayed stream only.
      warm  LRU whose cache is PRE-FILLED at load time with the most popular
            experts (from `warm`, a dict layer -> ranked expert ids fitted on a
            disjoint warmup slice). This is cache warming as a serving stack
            does it, and it costs one sequential read at startup.
      pin   `pin_frac` of each layer's slots are RESERVED for the warm set and
            never evicted; the remainder runs LRU. pin_frac=0 is warm-start LRU,
            pin_frac=1 is pure static pinning.

    `eval_from` skips the first N tokens when counting, so a policy fitted on a
    warmup slice is never scored on it.
    """
    T, L, k = ev.shape
    caps = split_capacity(cap_total, L) if scope == "per_layer" else [cap_total] * L
    hits = seen = 0
    for l in range(L):
        cp = caps[l]
        if cp <= 0:
            seen += (T - eval_from) * k
            continue
        n_pin = min(cp, int(round(pin_frac * cp)))
        pinned = set(warm[l][:n_pin]) if (warm is not None and n_pin) else set()
        cache = OrderedDict()
        if warm is not None:                      # pre-fill the free slots too
            for e in warm[l][:cp]:
                if e not in pinned:
                    cache[e] = None
        freq = {}
        col = ev[:, l, :]
        for t in range(T):
            keys = col[t].tolist()
            counting = t >= eval_from
            miss = []
            for key in keys:
                if counting:
                    seen += 1
                if key in pinned:
                    if counting:
                        hits += 1
                elif key in cache:
                    if counting:
                        hits += 1
                    cache.move_to_end(key)
                else:
                    miss.append(key)
                freq[key] = freq.get(key, 0) + 1
            free = cp - len(pinned)
            for key in miss:
                # free == 0 means every slot is pinned: nothing to evict into.
                while free > 0 and len(cache) >= free:
                    if policy == "lfu":
                        victim = min(cache, key=lambda e: (freq.get(e, 0),))
                        del cache[victim]
                    else:
                        cache.popitem(last=False)
                if free > 0:
                    cache[key] = None
    return hits, seen


def warm_sets(traces, num_experts, warmup_frac):
    """Expert ids per layer, ranked by frequency over the FIRST warmup_frac of
    tokens only, offset into the global key space. Fitted on a slice that the
    evaluation then skips, so nothing is scored on the data it was chosen on."""
    layers = sorted(traces)
    T = min(traces[l].shape[0] for l in layers)
    split = max(1, int(T * warmup_frac))
    out = {}
    for r, l in enumerate(layers):
        c = np.bincount(traces[l][:split].ravel().astype(np.int64),
                        minlength=num_experts)
        out[r] = (np.argsort(c)[::-1] + r * num_experts).tolist()
    return out, split


def corrupt_routing(ev, accuracy, num_experts, seed=0):
    """A PREDICTED view of the routing: each expert is right with probability
    `accuracy`, else replaced by a uniformly random expert of the same layer.

    accuracy=1.0 is a perfect oracle (or, equivalently, an accepted speculative
    draft, whose routing is exact by construction). accuracy=0.0 is no
    information at all. Layer offsets are preserved so a corrupted id never
    escapes its own layer's expert range.
    """
    rng = np.random.default_rng(seed)
    T, L, k = ev.shape
    out = ev.copy()
    flip = rng.random((T, L, k)) >= accuracy
    for l in range(L):
        m = flip[:, l, :]
        if m.any():
            col = out[:, l, :]
            col[m] = l * num_experts + rng.integers(0, num_experts, size=int(m.sum()))
            out[:, l, :] = col
    return out


def lookahead_hits(ev, cap_total, horizon, scope="per_layer", ev_hat=None,
                   mode="rank"):
    """LRU eviction improved by a BOUNDED lookahead of `horizon` tokens.

    horizon=None is exactly Belady, which tests/test_gates.py asserts as an
    identity. horizon=0 is NOT "no information": the engine is executing the
    current token, so it always knows that token's own expert set, and H=0 is
    therefore event-atomic (measured within ~1% of atomic LRU, far above
    sequential LRU). The tests assert the bracketing -- sequential LRU <= any
    horizon <= Belady -- and monotonicity in the horizon, rather than an
    identity that does not hold.

    WHERE THE LOOKAHEAD COMES FROM, and what happens when it is wrong:

      Multi-token verification supplies it EXACTLY, and it is worth being
      precise about why, because the obvious explanation is wrong. It is not
      that the draft model routed those tokens: a self-draft restricted to
      cache-resident experts is a different model, so its routing is a
      prediction of the target's. The exactness comes from the shape of the
      verification pass. Verifying W tokens is ONE batched forward pass, so
      layer l routes all W positions in a single matmul -- and that happens
      before layer l's expert weights are touched. The engine therefore reads
      the true top-k sets for W tokens off the router it has just run. The
      horizon is exactly the window and not one token more: nothing is yet
      known about the next window's layer-l needs.

      `ev_hat` models the cheaper alternative -- a routing PREDICTOR -- by
      letting eviction see a corrupted view of the future while hits are still
      counted against the true one.

      A wrong eviction spends no bandwidth on bytes nobody wanted, which is the
      direct cost that sinks a wrong PREFETCH (Budgeting Bytes 2609.04238; WiSP
      2606.21868; llama.cpp discussion #24528). It does NOT follow that a bad
      predictor is harmless here, and an earlier draft of this docstring claimed
      it did. Measured: under mode="rank" a predictor at accuracy 0 scores 0.095
      against LRU's 0.273 at a 10% cache, because ranking by a wrong prediction
      discards recency, which is itself real information. Break-even is accuracy
      ~0.70.

      `mode` is the fix, and which one to use depends on where the lookahead
      came from:

        "rank"     evict the farthest predicted next use. Optimal with an EXACT
                   lookahead -- which is what a batched verification pass
                   supplies over its own window -- and it reaches Belady at a
                   horizon of ~4 tokens.
        "protect"  the predictor may only VETO a candidate, never propose one;
                   recency still does the ranking. Strictly worse at accuracy
                   1.0 (0.379 vs 0.439) and far more robust: it still beats LRU
                   at accuracy 0.3. Use it when the lookahead is a PREDICTION.

    The engine always knows the CURRENT token's routing exactly, because it is
    executing that layer, so next-use distances inside the current token are
    taken from `ev` and only distances beyond it come from `ev_hat`.
    """
    T, L, k = ev.shape
    if ev_hat is None:
        ev_hat = ev
    INF = float("inf")
    hits = 0
    if scope == "global":
        # ONE pool over the token-major stream. The horizon is then counted in
        # EVENTS (layer-steps), not tokens, because that is the unit a global
        # cache advances by: a within-token, cross-layer predictor supplies a
        # horizon of a few events without any drafter, which a per-layer cache
        # cannot use at all.
        streams = [(ev.reshape(-1).tolist(), ev_hat.reshape(-1).tolist(), cap_total)]
    elif scope == "per_layer":
        caps = split_capacity(cap_total, L)
        streams = [(ev[:, l, :].reshape(-1).tolist(),
                    ev_hat[:, l, :].reshape(-1).tolist(), caps[l])
                   for l in range(L)]
    else:
        raise ValueError(f"scope must be 'global' or 'per_layer', got {scope!r}")

    for seq, hat, cp in streams:
        if cp <= 0:
            continue
        n = len(seq)
        # next occurrence of the TRUE key seq[i], strictly after i, in each of
        # the true and the predicted stream
        nxt_t, nxt_h = [0] * n, [0] * n
        last_t, last_h = {}, {}
        for i in range(n - 1, -1, -1):
            nxt_t[i] = last_t.get(seq[i], n + i)   # distinct far-future sentinel
            nxt_h[i] = last_h.get(seq[i], n + i)
            last_t[seq[i]] = i
            last_h[hat[i]] = i
        # per_layer: one event per token, so horizon*k accesses == horizon tokens.
        # global: k accesses per event, so horizon*k accesses == horizon EVENTS.
        span = n if horizon is None else horizon * k
        cache = OrderedDict()          # key -> (next true use, next predicted use)
        for i, key in enumerate(seq):
            tok_end = (i // k + 1) * k
            if key in cache:
                hits += 1
                cache.move_to_end(key)
            elif len(cache) >= cp:
                victim, best = None, -1.0
                for e, (vt, vh) in cache.items():   # iterates LRU-most first
                    if vt < tok_end:
                        d = float(vt - i)   # inside the token being executed: known
                    else:
                        d = INF if vh - i > span else float(vh - i)
                    if mode == "protect":
                        if d == INF:
                            victim = e     # LRU-most entry the horizon does not want
                            break
                        if victim is None and vt >= tok_end:
                            victim = e     # fallback: LRU-most that is not needed now
                    elif d > best:         # strictly greater keeps the LRU-most entry
                        victim, best = e, d
                        if d == INF:
                            break          # nothing beats "not needed in the horizon"
                if victim is None:
                    victim = next(iter(cache))
                del cache[victim]
            cache[key] = (nxt_t[i], nxt_h[i])
            cache.move_to_end(key)
    return hits


def simulate(traces, num_experts, fractions, max_tokens=None, belady=True,
             scopes=("per_layer",), replays=("atomic",)):
    ev, T, L, k = event_stream(traces, num_experts, max_tokens)
    total_experts = L * num_experts
    n_req = T * L * k
    out = []
    for f in fractions:
        cap = max(1, int(round(f * total_experts)))
        caps = split_capacity(cap, L)
        row = {"cache_fraction": f, "cache_experts": cap,
               "cap_per_layer_min": min(caps), "cap_per_layer_max": max(caps),
               # rho = per-layer slots / top_k = f * E / k, exact (not the
               # integer-floored min, which aliases adjacent fractions).
               "rho_per_layer": f * num_experts / k,
               "rho_per_layer_min_int": min(caps) / k,
               "floor_hit": cap / total_experts}
        for scope in scopes:
            for replay in replays:
                tag = f"{scope}_{replay}"
                h = lru_hits_events(ev, cap, scope=scope, replay=replay)
                row[f"lru_hit__{tag}"] = h / n_req
                row[f"lru_misses_per_token__{tag}"] = (n_req - h) / T
            if belady:
                b = belady_hits_scoped(ev, cap, scope=scope)
                row[f"belady_hit__{scope}"] = b / n_req
                row[f"belady_misses_per_token__{scope}"] = (n_req - b) / T
        out.append(row)
    return {"tokens": T, "moe_layers": L, "top_k": k, "num_experts": num_experts,
            "requests": int(n_req), "scopes": list(scopes), "replays": list(replays),
            "f_crit": k / num_experts, "rows": out}


def _crit_fractions(k, E, n_layers):
    """Fractions bracketing f_crit = k/E, where a per-layer cache first holds a
    whole token's working set. Pre-registered as multiples of f_crit so the grid
    is fixed by the geometry, not chosen after seeing the curve (CLAUDE.md §6.4)."""
    fc = k / E
    mult = [0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 3.0, 4.0]
    return sorted({round(min(1.0, m * fc), 6) for m in mult})


def main():
    p = argparse.ArgumentParser(description="expert-cache simulation from routing traces")
    p.add_argument("npz")
    p.add_argument("--fractions", default="0.05,0.1,0.2,0.3,0.5")
    p.add_argument("--fractions-around-crit", action="store_true",
                   help="sweep multiples of f_crit = k/E instead (pre-registered grid)")
    p.add_argument("--max-tokens", type=int, default=None)
    p.add_argument("--no-belady", action="store_true")
    p.add_argument("--scope", default="per_layer", choices=["per_layer", "global"])
    p.add_argument("--both-scopes", action="store_true",
                   help="report per_layer AND global, so the retracted global number stays visible")
    p.add_argument("--replay", default="atomic", choices=["atomic", "sequential"])
    p.add_argument("--both-replays", action="store_true")
    p.add_argument("--lookahead", default=None,
                   help="comma-separated lookahead horizons in TOKENS to sweep, e.g. "
                        "0,1,2,4,8,16. 0 is sequential LRU and an empty horizon is "
                        "Belady, so the curve says how many tokens of future routing "
                        "an engine needs to reach the offline optimum.")
    p.add_argument("--pred-accuracy", default=None,
                   help="comma-separated predictor accuracies to sweep alongside --lookahead, "
                        "e.g. 1.0,0.9,0.8,0.7. Eviction sees the corrupted routing; hits are "
                        "counted against the true one. Reports the break-even accuracy at "
                        "which a predictor stops beating plain LRU.")
    p.add_argument("--controls", action="store_true",
                   help="also simulate the shuffled and uniform controls, decomposing the hit "
                        "rate into floor + popularity skew + recency")
    p.add_argument("--seed", type=int, default=0, help="seed for the controls")
    p.add_argument("--out-name", default=None, help="basename override for the artifact")
    p.add_argument("--out-dir", default=None,
                   help="write here instead of results/<date>/ (use for tests and dry runs)")
    a = p.parse_args()
    z = np.load(a.npz)
    traces = {int(kk[1:]): z[kk] for kk in z.files if kk.startswith("L")}
    E = int(z["num_experts"])
    k = next(iter(traces.values())).shape[1]
    if a.fractions_around_crit:
        fracs = _crit_fractions(k, E, len(traces))
    else:
        fracs = [float(x) for x in a.fractions.split(",")]
    scopes = ("per_layer", "global") if a.both_scopes else (a.scope,)
    replays = ("atomic", "sequential") if a.both_replays else (a.replay,)
    res = simulate(traces, E, fracs, a.max_tokens, belady=not a.no_belady,
                   scopes=scopes, replays=replays)
    res["source"] = os.path.abspath(a.npz)
    if a.controls:
        res["controls"] = {"seed": a.seed}
        for cname, ctrace in (("shuffled", shuffle_control(traces, a.seed)),
                              ("uniform", uniform_control(traces, E, a.seed))):
            res["controls"][cname] = simulate(ctrace, E, fracs, a.max_tokens,
                                              belady=not a.no_belady,
                                              scopes=scopes, replays=replays)["rows"]
    print(f"{res['tokens']} tokens x {res['moe_layers']} MoE layers x "
          f"top-{res['top_k']} of {E} experts    f_crit = k/E = {res['f_crit']:.4f}")
    cols = [(f"lru_hit__{s}_{r}", f"LRU/{s[:3]}/{r[:3]}") for s in scopes for r in replays]
    cols += [(f"belady_hit__{s}", f"Belady/{s[:3]}") for s in scopes if not a.no_belady]
    hdr = f"{'cache':>6s} {'rho':>5s} {'floor':>7s}" + "".join(f" {t:>15s}" for _, t in cols)
    print(hdr)
    for r in res["rows"]:
        line = f"{r['cache_fraction']:6.3f} {r['rho_per_layer']:5.2f} {r['floor_hit']:7.3f}"
        line += "".join(f" {r.get(c, float('nan')):15.3f}" for c, _ in cols)
        print(line)
    if a.controls:
        print()
        print("decomposition of the hit rate (scope=per_layer, replay=atomic):")
        print(f"{'cache':>6} {'floor':>7} {'=uniform':>9} {'+skew':>8} {'+recency':>10} "
              f"{'=real':>8}   {'Belady real':>12} {'Belady unif':>12}")
        for i, r in enumerate(res["rows"]):
            u = res["controls"]["uniform"][i].get("lru_hit__per_layer_atomic")
            sh = res["controls"]["shuffled"][i].get("lru_hit__per_layer_atomic")
            rl = r.get("lru_hit__per_layer_atomic")
            if None in (u, sh, rl):
                continue
            bu = res["controls"]["uniform"][i].get("belady_hit__per_layer")
            br = r.get("belady_hit__per_layer")
            nan = float("nan")
            print(f"{r['cache_fraction']:6.3f} {r['floor_hit']:7.3f} {u:9.3f} {sh - u:+8.3f} "
                  f"{rl - sh:+10.3f} {rl:8.3f}   "
                  f"{(br if br is not None else nan):12.3f} "
                  f"{(bu if bu is not None else nan):12.3f}")
        print("  The cache fraction is the correct null for LRU -- the uniform column")
        print("  reproduces it -- but NOT for Belady: an offline optimum beats the fraction")
        print("  even on locality-free routing, so a Belady-minus-fraction gap is not a")
        print("  measure of exploitable locality and overstates it by the uniform column.")
    if a.lookahead:
        hs = [int(x) for x in a.lookahead.split(",")]
        ev_la, T_la, L_la, k_la = event_stream(traces, E, a.max_tokens)
        n_la = T_la * L_la * k_la
        res["lookahead"] = {"horizons_tokens": hs, "rows": []}
        print()
        print("hit rate vs LOOKAHEAD HORIZON (tokens of future routing already known).")
        print("Verifying W tokens in one batched pass supplies this exactly: layer l")
        print("routes all W positions before it touches layer l's expert weights.")
        hdr = f"{'cache':>6} {'LRU':>7}"
        hdr += "".join(f"{'H=' + str(h):>8}" for h in hs)
        hdr += f"{'Belady':>9}   gap closed by the largest H"
        print(hdr)
        for f in fracs:
            cap = max(1, int(round(f * L_la * E)))
            lru = lru_hits_events(ev_la, cap, "per_layer", "atomic") / n_la
            bel = belady_hits_scoped(ev_la, cap, "per_layer") / n_la
            xs = [lookahead_hits(ev_la, cap, h) / n_la for h in hs]
            closed = ((xs[-1] - lru) / (bel - lru)) if bel > lru else float("nan")
            line = f"{f:6.3f} {lru:7.3f}"
            line += "".join(f"{x:8.3f}" for x in xs)
            line += f"{bel:9.3f}   {100 * closed:5.0f}%"
            print(line)
            res["lookahead"]["rows"].append(
                {"cache_fraction": f, "lru_atomic": lru, "belady": bel,
                 "by_horizon": dict(zip(map(str, hs), xs)),
                 "fraction_of_belady_gap_closed": closed})
        if a.pred_accuracy:
            accs = [float(x) for x in a.pred_accuracy.split(",")]
            H = max(hs) if hs else 4
            res["prediction"] = {"horizon_tokens": H, "accuracies": accs, "rows": []}
            print()
            print(f"the same eviction rule at horizon {H}, driven by an IMPERFECT predictor.")
            print("Eviction sees the corrupted routing; hits are counted on the true trace.")
            print("rank uses the predicted ordering; protect lets the predictor only veto,")
            print("leaving recency to rank. rank wins with an exact lookahead (a draft);")
            print("protect is the one that survives a wrong predictor.")
            hdr = f"{'cache':>6} {'mode':>8} {'LRU':>7}"
            hdr += "".join(f"{format(x, '.2f'):>7}" for x in accs)
            hdr += f"{'Belady':>8}   break-even"
            print(hdr)
            for f in fracs:
                cap = max(1, int(round(f * L_la * E)))
                lru = lru_hits_events(ev_la, cap, "per_layer", "atomic") / n_la
                bel = belady_hits_scoped(ev_la, cap, "per_layer") / n_la
                row = {"cache_fraction": f, "lru_atomic": lru, "belady": bel}
                for mode in ("rank", "protect"):
                    ys = []
                    for acc in accs:
                        hat = None if acc >= 1.0 else corrupt_routing(ev_la, acc, E, a.seed)
                        ys.append(lookahead_hits(ev_la, cap, H, ev_hat=hat,
                                                 mode=mode) / n_la)
                    be = next((acc for acc, y in zip(accs, ys) if y < lru), None)
                    line = f"{f:6.3f} {mode:>8} {lru:7.3f}"
                    line += "".join(f"{y:7.3f}" for y in ys)
                    line += f"{bel:8.3f}   {(format(be, '.2f') if be is not None else 'never'):>9}"
                    print(line)
                    row[mode] = {"by_accuracy": dict(zip(map(str, accs), ys)),
                                 "break_even_accuracy": be}
                res["prediction"]["rows"].append(row)
    name = a.out_name or os.path.splitext(os.path.basename(a.npz))[0].replace("traces_", "cache_")
    if a.out_dir:
        os.makedirs(a.out_dir, exist_ok=True)
        out = os.path.join(a.out_dir, name + ".json")
    else:
        sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
        from _paths import write_path
        out = write_path(name + ".json")
    with open(out, "w", encoding="utf-8") as f:
        json.dump(res, f, indent=1)
    print("wrote", out)


if __name__ == "__main__":
    main()


In [ ]:
# Pinned because a gate's numbers are only reproducible against a fixed
# dependency set (CLAUDE.md section 6). Versions are printed, not assumed.
!pip install -q "transformers==4.56.2" accelerate datasets
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}, {p.total_memory / 1e9:.1f} GB, bf16={torch.cuda.is_bf16_supported()}")

# Checked here, before anything expensive. OLMoE-1B-7B in float16 is ~14 GB of
# weights: on CPU this would not merely be slow, it would be a different
# measurement, and it would run for hours before saying so.
if torch.cuda.device_count() < 1:
    raise SystemExit(
        "\n*** NO GPU ATTACHED — nothing below this cell can run. ***\n"
        "Kaggle reports Accelerator: None.\n"
        "Fix: open the right-hand panel -> Session options -> Accelerator -> 'GPU T4 x2',\n"
        "wait for the session to restart, THEN Run All (or Save Version -> Save & Run All).\n"
        "The accelerator must be set BEFORE the run starts; changing it afterwards does\n"
        "not re-run the notebook.")


In [ ]:
import subprocess, sys, time

def run(*args, label):
    """Run a gate step and STOP the notebook if it fails.

    A bare `!command` in Jupyter reports a failure and carries on to the next
    cell, which is how a broken run ends as a zip file full of nothing. Raising
    here makes 'Run All' halt at the first real problem."""
    t0 = time.time()
    print(f"$ {' '.join(args)}", flush=True)
    rc = subprocess.run([sys.executable, *args]).returncode
    print(f"[{label}] exit={rc} in {time.time() - t0:.0f}s", flush=True)
    if rc != 0:
        raise SystemExit(f"{label} FAILED (exit {rc}) - stop here; later numbers mean nothing.")


In [ ]:
# Must print "0 failure(s)". Runs on CPU with tiny random models, so it costs
# nothing and it is the only thing standing between a bug and a plausible number.
run("traces_sparsity.py", "--selftest", label="G2/G3 self-test")


In [ ]:
# G2 + G3 on the stock checkpoint. ~20-40 min on T4 x2.
run("traces_sparsity.py", "--model", "allenai/OLMoE-1B-7B-0924",
    "--windows", "64", "--calib-windows", "16", "--seq-len", "512",
    "--out-dir", "/kaggle/working/out", label="G3 fidelity + G2 traces")


In [ ]:
# G2 offline half: replay the routing traces through an expert cache.
#
# --controls decomposes the hit rate into floor + popularity skew + recency, so
#   the result is not "there is locality" but how much of it is which, against
#   a locality-free null.
# --both-scopes keeps the retracted shared-pool number visible beside the
#   per-layer one that replaces it.
# --lookahead sweeps how many tokens of FUTURE routing a cache needs to reach
#   the offline optimum. This is the constructive number, because a batched
#   verification pass over W tokens supplies exactly that much for free: layer l
#   routes all W positions in one matmul, before it touches layer l's experts.
# Cost note: the lookahead rule scans the cache on every eviction, so runtime
# grows with the cache size. Fractions are capped at 0.30 and horizons at 8 to
# keep this cell to a few minutes; nothing above those changed the conclusion
# when swept locally.
run("cache_sim.py", "/kaggle/working/out/traces_OLMoE-1B-7B-0924.npz",
    "--controls", "--both-scopes", "--both-replays",
    "--lookahead", "0,1,2,4,8", "--fractions", "0.05,0.10,0.125,0.20,0.30",
    "--out-dir", "/kaggle/working/out", label="G2 cache simulation")

# How far a NOISY lookahead (a predictor, rather than a draft) gets, and under
# which of the two eviction rules. Fewer fractions: this is the most expensive
# sweep in the notebook, and the break-even accuracy is what it exists to print.
run("cache_sim.py", "/kaggle/working/out/traces_OLMoE-1B-7B-0924.npz",
    "--lookahead", "4", "--pred-accuracy", "1.0,0.9,0.8,0.7,0.5,0.3,0.0",
    "--fractions", "0.10,0.20", "--max-tokens", "16384",
    "--out-name", "cache_pred_OLMoE-1B-7B-0924",
    "--out-dir", "/kaggle/working/out", label="G2c predictor-accuracy sweep")

# The pre-registered sweep across f_crit = k/E, the cache fraction at which a
# per-layer cache first holds one token's whole working set.
run("cache_sim.py", "/kaggle/working/out/traces_OLMoE-1B-7B-0924.npz",
    "--fractions-around-crit", "--both-replays",
    "--out-name", "cache_fcrit_OLMoE-1B-7B-0924",
    "--out-dir", "/kaggle/working/out", label="G2 critical-fraction sweep")


In [ ]:
# S9 -- the geometry-transfer assumption, which currently gates EVERY per-model
# tok/s figure in ARCHITECTURE.md.
#
# engine_target.py reads each model's hit rate off a curve measured on OLMoE's
# 64-expert top-8 routing and applies it at equal rho = per-layer slots / top_k.
# That the hit rate depends on geometry ONLY through rho is an assumption. To test
# it, collect a trace at a materially different EXPERT COUNT and compare the two
# curves at equal rho.
#
# CHOOSING THE MODEL. The requirement is a materially different EXPERT COUNT that
# actually fits here. Note --load-in-4bit shrinks what sits in VRAM, NOT what is
# downloaded: transformers fetches the full 16-bit weights and quantises on load.
# So the download size, not the GPU, is usually the binding constraint on Kaggle.
#
#   Qwen/Qwen1.5-MoE-A2.7B   E=60,  k=4   ~29 GB download   DEFAULT, fits
#   deepseek-ai/DeepSeek-V2-Lite  E=64, k=6  ~31 GB         needs trust_remote_code
#   Qwen/Qwen3-30B-A3B       E=128, k=8   ~61 GB download   the ideal target, and
#                                                           the one in our candidate
#                                                           table -- but the download
#                                                           will likely exceed Kaggle
#                                                           disk. Use a machine with
#                                                           more disk, or a
#                                                           pre-quantised repo.
#
# --densities 1.0 --no-floor skips the fidelity sweep, because traces are all S9
# needs; that makes this far cheaper than the G3 cell above.
#
# SET THIS TO True TO RUN IT. It is off by default so the notebook's default path
# stays the OLMoE reproduction.
RUN_S9 = False
S9_MODEL = "Qwen/Qwen1.5-MoE-A2.7B"

if RUN_S9:
    # Peek at the geometry from config.json BEFORE downloading any weights: if E
    # is not materially different from OLMoE's 64, this run cannot test anything
    # and there is no point spending the download.
    from transformers import AutoConfig
    _c = AutoConfig.from_pretrained(S9_MODEL, trust_remote_code=True)
    _E = getattr(_c, "num_experts", None) or getattr(_c, "n_routed_experts", None)
    _k = (getattr(_c, "num_experts_per_tok", None)
          or getattr(_c, "num_experts_per_token", None))
    print(f"{S9_MODEL}: E={_E}, k={_k}   (OLMoE is E=64, k=8)")
    print(f"rho = cache_fraction * E / k = cache_fraction * {(_E / _k) if _E and _k else '?'}")
    assert _E and _k, "could not read the expert geometry from this config"
    assert _E != 64 or _k != 8, (
        "same geometry as OLMoE -- this run cannot test the transfer assumption")

    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"],
                   check=False)
    run("traces_sparsity.py", "--model", S9_MODEL, "--load-in-4bit",
        "--trust-remote-code", "--densities", "1.0", "--no-floor",
        "--windows", "64", "--calib-windows", "4", "--seq-len", "512",
        "--out-dir", "/kaggle/working/out", label="S9 traces: " + S9_MODEL)
    tag = S9_MODEL.split("/")[-1]
    run("cache_sim.py", f"/kaggle/working/out/traces_{tag}.npz",
        "--controls", "--lookahead", "0,1,2,4,8",
        "--fractions-around-crit",
        "--out-dir", "/kaggle/working/out", label="S9 cache curve: " + tag)
    print("Compare this curve against the OLMoE one AT EQUAL RHO, not at equal cache")
    print("fraction: rho = cache_fraction * E / k, and E differs between them.")
else:
    print("S9 cell skipped. Set RUN_S9 = True to collect the second-geometry trace.")


In [ ]:
# Pack for download. Uses zipfile rather than the `zip` binary, which is not
# installed on every Kaggle image.
import os, zipfile
out, archive = "/kaggle/working/out", "/kaggle/working/moe_phone_out.zip"
files = sorted(os.listdir(out))
assert files, f"{out} is empty - an earlier cell failed."
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(os.path.join(out, f), f)
        print(f"  {f}  {os.path.getsize(os.path.join(out, f)) / 1e6:.2f} MB")
print(f"\nwrote {archive} ({os.path.getsize(archive) / 1e6:.2f} MB)")
print("Download it from the Output panel, then unpack into moe-phone/results/<today>/")
